# 02 — Baselines: análise da validação cruzada

Carrega os resultados de `outputs/` (gerados por `python pathoshift.py train ...`) e analisa
os folds, as curvas da CNN, o recall por paciente e os erros. Não mostra o teste externo
(ele está no notebook 03).

In [ ]:
# ===== Célula de ambiente (igual em todos os notebooks) =====
# Local (kernel .venv): usa o que já está em data/ e outputs/; não instala nem baixa nada.
# Colab (extensão do VS Code ou navegador): monta o Drive, clona/atualiza o repositório,
# instala as dependências e prepara o que este notebook precisa. Preencha REPO_URL para o Colab.
REPO_URL = "https://github.com/SEU_USUARIO/pathoshift.git"
BRANCH = "main"

import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/pathoshift")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ambiente import preparar
preparar(ROOT, precisa_dados=True, precisa_features=False, precisa_resultados=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import utils
from src.data import load_hf_dataset, read_image
from src.manifest import load_manifest, summarize

cfg = utils.load_config(ROOT / "configs" / "default.yaml")
utils.set_seed(cfg["seed"])
man = load_manifest(utils.resolve_path(cfg["paths"]["manifest"]))
ds = load_hf_dataset(cfg)

import json
OUT = utils.resolve_path(cfg["paths"]["outputs_dir"])
oof = {m: pd.read_csv(OUT / m / "cv_oof.csv").merge(man.drop(columns="fold"), on="id") for m in ["logreg", "cnn"]}

## 1. Métricas por patch em cada fold

In [ ]:
for m in ["logreg", "cnn"]:
    print(m)
    display(pd.read_csv(OUT / m / "cv_folds.csv", index_col=0).round(3))

## 2. CNN: conjunto de parada e curvas por época

In [ ]:
info = json.loads((OUT / "cnn" / "cv_info.json").read_text())
display(pd.DataFrame(info).T[["pacientes_parada_com_tumor", "melhor_epoca", "epocas_treinadas", "segundos"]])
from IPython.display import Image
display(Image(str(OUT / "figuras" / "curvas_cnn.png")))

## 3. Recall de tumor por paciente (o que explica o desvio entre folds)

In [ ]:
def recall_por_paciente(df):
    t = df[df["label"] == 1]
    return t.groupby(["fold", "centre_id", "patient_id"])["prob"].agg(
        n="size", prob_media="mean", recall=lambda p: (p >= 0.5).mean())

pd.concat({m: recall_por_paciente(oof[m]) for m in oof}, axis=1).round(3)

## 4. Falsos positivos e falsos negativos mais confiantes

In [ ]:
for m in ["logreg", "cnn"]:
    display(Image(str(OUT / "figuras" / f"erros_{m}.png")))